In [3]:
from pathlib import Path
import subprocess
import sys

data_dir = Path("data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

try:
    subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            "xavierberge/hospital-emergency-dataset",
            "-p",
            str(data_dir),
            "--unzip",
        ],
        check=True,
    )
    print(f"Downloaded dataset to {data_dir}")
except FileNotFoundError:
    print("Kaggle CLI not found. Install it with `pip install kaggle` and configure ~/.kaggle/kaggle.json.")
    sys.exit(1)
except subprocess.CalledProcessError as exc:
    print(f"Kaggle download failed with exit code {exc.returncode}.")
    sys.exit(exc.returncode)

Downloaded dataset to data\raw


# 1. Study Design and Problem Formulation

## Project Objective

The goal of this project is to formulate Emergency Room demand forecasting as a multi-horizon time-series modeling problem. Instead of treating ER forecasting as one single prediction task, this study separates the problem into distinct forecasting objectives that support different hospital planning decisions:

- **Short-term forecasting** supports immediate operational decisions such as hourly staffing, bed readiness, triage desk coverage, and same-day surge preparation.
- **Medium-term forecasting** supports weekly and monthly planning decisions such as shift scheduling, seasonal resource allocation, and preparation for recurring demand patterns.
- **Long-term forecasting** supports strategic planning decisions such as annual capacity planning, workforce budgeting, and identifying broader changes in patient demand.

The project therefore defines separate mathematical forecasting tasks for ER patient volume and patient acuity mix across short, medium, and long-term horizons.

---

## Raw Observation Unit

The raw dataset is structured at the patient-visit level. Each row represents one patient encounter in the emergency department. The most important fields for the study design are:

- `Patient Admission Date`: timestamp of the patient visit or arrival
- `Patient Id`: unique patient-level identifier that can be counted to measure visit volume
- `Patients CM`: binary indicator that can be used as a proxy for higher-complexity or higher-acuity cases
- `Patient Admission Flag`: whether the patient was admitted
- `Patient Waittime`: wait time associated with the encounter
- demographic fields such as age, gender, and race
- referral fields such as `Department Referral`

Because forecasting models require observations indexed by time, the patient-level records must be aggregated into regular time intervals such as hourly, daily, weekly, or monthly periods.

---

## Core Forecasting Targets

This project has two primary forecasting targets: ER volume and triage acuity mix.

### 1. ER Volume Target

ER volume measures how many patients arrive during a specific time interval.

Let:

- `t` represent a time interval, such as an hour, day, week, or month
- `y_t` represent the total number of ER patient arrivals during interval `t`

The volume target is defined as:

```text
y_t = count of patient visits during time interval t
```

In this dataset, `y_t` can be calculated by counting `Patient Id` records after grouping by the selected time frequency.

### 2. Triage Acuity Target

The dataset does not contain a direct clinical triage score such as Emergency Severity Index levels 1 through 5. Therefore, this project must define a practical acuity proxy using the available fields.

The best available proxy is `Patients CM`, which is a binary indicator. For this study:

- `Patients CM = 1` is treated as a higher-complexity or higher-acuity patient encounter
- `Patients CM = 0` is treated as a lower-complexity encounter

Let:

- `a_t` represent the number of higher-acuity encounters during interval `t`
- `p_t` represent the share of higher-acuity encounters during interval `t`

The acuity targets are defined as:

```text
a_t = count of encounters where Patients CM = 1 during time interval t

p_t = a_t / y_t
```

This creates two possible acuity modeling targets:

- **High-acuity count**, which forecasts the number of complex patients expected
- **High-acuity rate**, which forecasts the proportion of ER demand expected to be complex

The high-acuity count is useful for staffing and resource planning because it estimates workload volume. The high-acuity rate is useful for understanding whether the composition of demand is changing over time.

---

## General Mathematical Forecasting Formulation

The forecasting problem can be written as a supervised time-series task. For each forecast horizon `h`, the model uses historical information available up to time `t` to predict a future target at time `t + h`.

For ER volume forecasting:

```text
y_hat_{t+h} = f(X_t, X_{t-1}, ..., X_{t-k})
```

Where:

- `y_hat_{t+h}` is the predicted ER arrival volume for future horizon `h`
- `X_t` is the feature set available at time `t`
- `k` is the lookback window used to capture prior temporal behavior
- `f` is the forecasting model

For acuity forecasting:

```text
a_hat_{t+h} = g(X_t, X_{t-1}, ..., X_{t-k})

p_hat_{t+h} = q(X_t, X_{t-1}, ..., X_{t-k})
```

Where:

- `a_hat_{t+h}` is the predicted count of higher-acuity patients
- `p_hat_{t+h}` is the predicted proportion of higher-acuity patients
- `g` and `q` may be separate models or multi-output extensions of the same model family

This structure allows the project to treat volume and acuity as related but distinct modeling problems.

---

## Multi-Horizon Forecasting Tasks

The study is divided into three forecasting horizons. Each horizon has a different time scale, business purpose, target definition, and modeling strategy.

| Horizon | Forecast Window | Time Aggregation | Volume Target | Acuity Target | Planning Use |
|---|---:|---|---|---|---|
| Short-term | 1 to 72 hours | Hourly or daily | Hourly or daily ER arrivals | High-acuity count or high-acuity rate | Staffing, triage coverage, bed readiness |
| Medium-term | 1 to 6 months | Weekly or monthly | Weekly or monthly ER arrivals | Monthly high-acuity count or share | Seasonal planning, shift schedules, resource allocation |
| Long-term | 6 to 12+ months | Monthly or quarterly | Long-run ER demand trend | Long-run acuity trend | Capacity planning, budgeting, workforce strategy |

---

## Short-Term Forecasting Objective

The short-term task focuses on high-frequency ER demand over the next 1 to 72 hours. This is the most operationally urgent forecasting problem because it supports near-real-time staffing and patient-flow decisions.

### Short-Term Volume Task

```text
Forecast y_{t+h} for h = 1, 2, ..., 72 hours
```

The target is the number of ER arrivals expected in each future hour or day.

Examples:

- number of arrivals expected next hour
- number of arrivals expected during the next 12 hours
- number of arrivals expected over the next 72 hours

### Short-Term Acuity Task

```text
Forecast a_{t+h} or p_{t+h} for h = 1, 2, ..., 72 hours
```

The target is either the count or proportion of higher-complexity encounters expected in the near future.

This horizon should capture:

- hour-of-day patterns
- day-of-week patterns
- weekend versus weekday differences
- sudden arrival spikes
- short-term changes in patient complexity

Potential model families for this horizon include LightGBM, random forests, gradient boosting models, and recurrent neural networks such as LSTM models if the dataset size supports deep learning.

---

## Medium-Term Forecasting Objective

The medium-term task focuses on demand over the next several weeks or months. This task is less about immediate hourly volatility and more about recurring seasonal structure.

### Medium-Term Volume Task

```text
Forecast y_{t+h} for h = 1, 2, ..., 6 months
```

The target is weekly or monthly ER arrival volume.

Examples:

- total ER arrivals expected next week
- total ER arrivals expected next month
- expected demand during high-season periods

### Medium-Term Acuity Task

```text
Forecast monthly a_{t+h} or p_{t+h}
```

The objective is to estimate whether the number or share of higher-acuity patients is expected to increase during upcoming weeks or months.

This horizon should capture:

- monthly seasonality
- recurring demand cycles
- flu-season or winter-season demand effects if visible in the data
- changes in patient mix over time

Potential model families for this horizon include SARIMAX, Prophet, exponential smoothing, and gradient boosting models using calendar and lag features.

---

## Long-Term Forecasting Objective

The long-term task focuses on broader trend behavior over 6 to 12 or more months. This task supports strategic hospital planning rather than immediate operations.

### Long-Term Volume Task

```text
Forecast y_{t+h} for h = 6 to 12+ months
```

The target is monthly or quarterly ER demand.

Examples:

- expected annualized ER demand
- expected quarterly patient volume
- long-run growth or decline in ER arrivals

### Long-Term Acuity Task

```text
Forecast long-run a_{t+h} or p_{t+h}
```

The target is the long-term trend in complex patient encounters, either as a count or as a proportion of total ER demand.

This horizon should capture:

- long-term demand trend
- gradual changes in patient complexity
- demographic shifts visible through age, gender, or race distributions
- structural changes in patient volume

Potential model families for this horizon include structural time-series models, SARIMAX, Prophet, and trend-based regression models.

---

## Final Problem Statement

This study formulates ER demand forecasting as a multi-horizon, multi-target time-series problem. The first target is patient arrival volume, defined as the number of ER visits within a fixed time interval. The second target is triage acuity mix, represented by the count or proportion of higher-complexity encounters using `Patients CM` as an acuity proxy.

The final modeling objective is to learn forecasting functions that predict future ER volume and future higher-acuity demand across short, medium, and long-term horizons:

```text
Volume forecasting target: y_{t+h}

Acuity count forecasting target: a_{t+h}

Acuity rate forecasting target: p_{t+h} = a_{t+h} / y_{t+h}
```

These forecasting tasks will support both operational and strategic decision-making by estimating not only how many patients are expected, but also how complex the expected patient demand may be.
